In [108]:
import pandas as pd
import numpy as np
import csv

In [119]:
data = pd.read_csv('../临时文件/8月勇者数据.csv',
                   index_col=False,
                   names=['id','事件名','时间','属性'],
                   encoding="utf-8",
                   on_bad_lines='skip',
                   engine='python'
                  )

In [120]:
data.shape

(1695192, 4)

In [123]:
data['详细日期']=pd.to_datetime(data['时间'], unit='ms')

In [124]:
data.query('事件名=="In_appPurchases_BuySuccess"')

,id,事件名,时间,属性,详细日期
3395,15iRDdScYLkVuGmCHRNPbY,In_appPurchases_BuySuccess,1723735995047,"{""#currency"": ""JPY"", ""#currency_ex"": ""USD"", ""#...",2024-08-15 15:33:15.047
15279,1889SFnghZUt8d0jR4y1Aj,In_appPurchases_BuySuccess,1723320851494,"{""#currency"": ""JPY"", ""#currency_ex"": ""USD"", ""#...",2024-08-10 20:14:11.494
19940,1889SFnghZUt8d0jR4y1Aj,In_appPurchases_BuySuccess,1723689009895,"{""#currency"": ""JPY"", ""#currency_ex"": ""USD"", ""#...",2024-08-15 02:30:09.895
21429,1889SFnghZUt8d0jR4y1Aj,In_appPurchases_BuySuccess,1723777319652,"{""#currency"": ""JPY"", ""#currency_ex"": ""USD"", ""#...",2024-08-16 03:01:59.652
29172,1889SFnghZUt8d0jR4y1Aj,In_appPurchases_BuySuccess,1724986229746,"{""#currency"": ""JPY"", ""#currency_ex"": ""USD"", ""#...",2024-08-30 02:50:29.746
...,...,...,...,...,...
1691234,3VhSKapuZUijMlGPZDGK7B,In_appPurchases_BuySuccess,1723683442457,"{""#currency"": ""JPY"", ""#currency_ex"": ""USD"", ""#...",2024-08-15 00:57:22.457
1692355,3VhSKapuZUijMlGPZDGK7B,In_appPurchases_BuySuccess,1723768240335,"{""#currency"": ""JPY"", ""#currency_ex"": ""USD"", ""#...",2024-08-16 00:30:40.335
1693367,3VhSKapuZUijMlGPZDGK7B,In_appPurchases_BuySuccess,1723853234376,"{""#currency"": ""JPY"", ""#currency_ex"": ""USD"", ""#...",2024-08-17 00:07:14.376
1693600,3VhSKapuZUijMlGPZDGK7B,In_appPurchases_BuySuccess,1723855374375,"{""#currency"": ""JPY"", ""#currency_ex"": ""USD"", ""#...",2024-08-17 00:42:54.375


### 用户内购前 总游戏时长、失败关卡、失败次数

In [139]:
# 用户首次内购事件发生时间
data_ng = data.query('事件名=="In_appPurchases_BuySuccess"').groupby(by=['id'],as_index=False)['详细日期'].min()

In [140]:
data_ng.columns = ['id','首次付费时间']

In [141]:
# 用户闯关数据 
data_chapter = data.query('事件名=="Game_Level_Finish"')
data_chapter = data_ng.merge(data_chapter,how='left',on ='id').query('详细日期<首次付费时间')

In [142]:
data_chapter['属性'] = data_chapter['属性'].apply(eval)

In [143]:
data_chapter['时长']=data_chapter['属性'].apply(lambda x:x.get('level_time',None)/60)

In [144]:
data_chapter['关卡'] = data_chapter['属性'].apply(lambda x:x.get('level_name',None))

In [145]:
data_chapter['日期'] =  pd.to_datetime(data_chapter['时间'], unit='ms').dt.strftime('%Y-%m-%d')

In [146]:
data_chapter['是否通关'] = data_chapter['属性'].apply(lambda x: '是' if x.get('if_clearance',None) ==1 else '否')

In [147]:
data_chapter['闯关顺序']=data_chapter.groupby(by=['id'])['详细日期'].rank(ascending=False,method='dense')

In [148]:
data_chapter1= data_chapter.query('闯关顺序==1')[['id','关卡']]

In [149]:
data_chapter1.columns = ['id','付费前闯关关卡']

In [150]:
data_chapter=data_chapter.merge(data_chapter1,how='left',on = 'id')

In [152]:
data_chapter.query('id=="15iRDdScYLkVuGmCHRNPbY"')

,id,首次付费时间,事件名,时间,属性,详细日期,时长,关卡,日期,是否通关,闯关顺序,付费前闯关关卡
0,15iRDdScYLkVuGmCHRNPbY,2024-08-15 15:33:15.047,Game_Level_Finish,1723694257761,"{'current_HP': '0', 'current_level': '2', 'cur...",2024-08-15 03:57:37.761,4.516667,ChapterName_1,2024-08-15,是,30.0,ChapterName_6
1,15iRDdScYLkVuGmCHRNPbY,2024-08-15 15:33:15.047,Game_Level_Finish,1723694634698,"{'current_HP': '0', 'current_level': '2', 'cur...",2024-08-15 04:03:54.698,4.000000,ChapterName_2,2024-08-15,是,29.0,ChapterName_6
2,15iRDdScYLkVuGmCHRNPbY,2024-08-15 15:33:15.047,Game_Level_Finish,1723695821351,"{'current_HP': '0', 'current_level': '3', 'cur...",2024-08-15 04:23:41.351,3.800000,ChapterName_1,2024-08-15,否,28.0,ChapterName_6
3,15iRDdScYLkVuGmCHRNPbY,2024-08-15 15:33:15.047,Game_Level_Finish,1723695821965,"{'current_HP': '0', 'current_level': '3', 'cur...",2024-08-15 04:23:41.965,3.800000,ChapterName_1,2024-08-15,否,27.0,ChapterName_6
4,15iRDdScYLkVuGmCHRNPbY,2024-08-15 15:33:15.047,Game_Level_Finish,1723696331604,"{'current_HP': '0', 'current_level': '3', 'cur...",2024-08-15 04:32:11.604,4.366667,ChapterName_1,2024-08-15,否,26.0,ChapterName_6
5,15iRDdScYLkVuGmCHRNPbY,2024-08-15 15:33:15.047,Game_Level_Finish,1723696845129,"{'current_HP': '0', 'current_level': '3', 'cur...",2024-08-15 04:40:45.129,4.750000,ChapterName_1,2024-08-15,否,25.0,ChapterName_6
6,15iRDdScYLkVuGmCHRNPbY,2024-08-15 15:33:15.047,Game_Level_Finish,1723697191966,"{'current_HP': '0', 'current_level': '3', 'cur...",2024-08-15 04:46:31.966,4.533333,ChapterName_3,2024-08-15,是,24.0,ChapterName_6
7,15iRDdScYLkVuGmCHRNPbY,2024-08-15 15:33:15.047,Game_Level_Finish,1723697927835,"{'current_HP': '0', 'current_power': '118', 'i...",2024-08-15 04:58:47.835,4.250000,ChapterName_4,2024-08-15,是,23.0,ChapterName_6
8,15iRDdScYLkVuGmCHRNPbY,2024-08-15 15:33:15.047,Game_Level_Finish,1723698575477,"{'current_HP': '0', 'current_level': '4', 'cur...",2024-08-15 05:09:35.477,4.483333,ChapterName_1,2024-08-15,是,22.0,ChapterName_6
9,15iRDdScYLkVuGmCHRNPbY,2024-08-15 15:33:15.047,Game_Level_Finish,1723699009594,"{'current_HP': '0', 'current_level': '4', 'cur...",2024-08-15 05:16:49.594,3.750000,ChapterName_2,2024-08-15,是,21.0,ChapterName_6


In [153]:
# 计算付费前关卡和失败次数
res0=data_chapter.query('关卡==付费前闯关关卡&是否通关=="否"').groupby(by=['id','付费前闯关关卡'],as_index=False)['关卡'].count()


In [154]:

#计算时长和天数
res1= data_chapter.groupby(by='id')[['时长','日期']].agg({
    '时长':'sum',
    '日期':'nunique',
}).reset_index()

In [155]:
res1.shape

(89, 3)

In [156]:
res = res1.merge(res0,how='left',on='id')

In [157]:
res.columns=['id','游戏总时长','游戏天数','付费前关卡','失败次数']

In [158]:
res.head()

,id,游戏总时长,游戏天数,付费前关卡,失败次数
0,15iRDdScYLkVuGmCHRNPbY,129.966667,1,NaN,NaN
1,1889SFnghZUt8d0jR4y1Aj,30.666667,1,ChapterName_5,1.0
2,18nr8nEuO6gvbfRXgIA7Ga,302.233333,3,ChapterName_23,6.0
3,1DUU4zXgJsOKNjxDydRYzC,22.000000,1,NaN,NaN
4,1HmkiDlV7dFU6ezekrZ5Yh,335.366667,7,ChapterName_18,2.0


### 游戏道具情况

In [121]:
# 宝石获取个数、有无强力宝石、抽卡情况、s、a、b角色个数

In [159]:
data_role = data.query('事件名=="role_get"')
data_role = data_ng.merge(data_role,how='left',on ='id').query('详细日期<首次付费时间')

In [160]:
data_role['角色名称_等级'] = data_role['属性'].apply(eval).apply(lambda x:x.get('name') +'_'+ str(x.get('quality')))

In [161]:
data_role.query('id=="1889SFnghZUt8d0jR4y1Aj"')

,id,首次付费时间,事件名,时间,属性,详细日期,角色名称_等级
14,1889SFnghZUt8d0jR4y1Aj,2024-08-10 20:14:11.494,role_get,1723312523893,"{""current_HP"": ""0"", ""current_level"": ""2"", ""cur...",2024-08-10 17:55:23.893,Blazewing_3
15,1889SFnghZUt8d0jR4y1Aj,2024-08-10 20:14:11.494,role_get,1723313248111,"{""current_HP"": ""0"", ""current_level"": ""3"", ""cur...",2024-08-10 18:07:28.111,Jessica_3


In [162]:
role_res=data_role.groupby(by=['id'],as_index=False)['角色名称_等级'].nunique()

In [169]:
role_res.head()

,id,角色名称_等级
0,15iRDdScYLkVuGmCHRNPbY,7
1,1889SFnghZUt8d0jR4y1Aj,2
2,18nr8nEuO6gvbfRXgIA7Ga,2
3,1DUU4zXgJsOKNjxDydRYzC,3
4,1HmkiDlV7dFU6ezekrZ5Yh,10


In [164]:
role_recruit = data.query('事件名=="role_recruit"')
role_recruit = data_ng.merge(role_recruit,how='left',on ='id').query('详细日期<首次付费时间')

In [165]:
role_recruit['抽卡次数'] = role_recruit['属性'].apply(eval).apply(lambda x: 10 if x.get('reason')=='Ten' else 1)

In [167]:
# 内购前抽卡总次数
gacha_res=role_recruit.groupby(by='id',as_index=False)['抽卡次数'].sum()

In [170]:
gacha_res.head()

,id,抽卡次数
0,15iRDdScYLkVuGmCHRNPbY,12
1,1889SFnghZUt8d0jR4y1Aj,2
2,18nr8nEuO6gvbfRXgIA7Ga,43
3,1DUU4zXgJsOKNjxDydRYzC,5
4,1HmkiDlV7dFU6ezekrZ5Yh,78


In [172]:
# 数据汇总
res=data_ng.merge(res1,how='left',on='id').merge(role_res,how='left',on='id').merge(gacha_res,how='left',on='id')

In [173]:
res.to_excel('用户付费前操作行为.xlsx')